# Config
Evaluates 3-fold CV predictions for HR connectivity reconstruction.
Adjust paths as needed.

In [ ]:
from pathlib import Path
import os
import json
import pickle

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr
from scipy.spatial.distance import jensenshannon
import networkx as nx
import matplotlib.pyplot as plt

from MatrixVectorizer import MatrixVectorizer
from model import BrainGCN_SR

# ---- User-editable paths ----
DATA_PATH = Path("data/hr_train.csv")
LR_PATH = Path("data/lr_train.csv")
CHECKPOINT_PATH = Path("outputs/checkpoints/best_model.pth")
PRED_PATHS = [
    Path("outputs/predictions_fold_1.csv"),
    Path("outputs/predictions_fold_2.csv"),
    Path("outputs/predictions_fold_3.csv"),
]

SPLITS_DIR = Path("splits")
OUTPUTS_DIR = Path("outputs")

SPLITS_PKL = SPLITS_DIR / "kfold3_seed42_indices.pkl"
SPLITS_JSON = SPLITS_DIR / "kfold3_seed42_indices.json"

# ---- Model hyperparameters (must match checkpoint) ----
MODEL_HIDDEN_DIMS = [256, 256, 256]
MODEL_EDGE_DIM = 128
MODEL_DROPOUT = 0.1
MODEL_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Pre/Post-processing switches ----
CLIP_TO_UNIT_INTERVAL = False  # set True to clip values to [0, 1]

# ---- Plot switches ----
USE_LOG_SCALE = False  # alternative scale for the single-figure bar plot
MAKE_SPLIT_PLOTS = False  # optional secondary plots with separate scales

# Reproducibility for CV
KFOLD_N_SPLITS = 3
KFOLD_SHUFFLE = True
KFOLD_RANDOM_STATE = 42

# Constants
N_ROI = 268
EPS = 1e-12

SPLITS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# 2. Load data

In [4]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Ground-truth file not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])
train_HR = df.to_numpy()
print("Loaded train_HR shape:", train_HR.shape)

Loaded train_HR shape: (167, 35778)


# 3. Create & save CV splits

In [5]:
kf = KFold(n_splits=KFOLD_N_SPLITS, shuffle=KFOLD_SHUFFLE, random_state=KFOLD_RANDOM_STATE)
splits = []

for fold_idx, (train_idx, test_idx) in enumerate(kf.split(train_HR), start=1):
    splits.append({
        "fold": fold_idx,
        "train_idx": train_idx,
        "test_idx": test_idx
    })

# Save pickle (numpy arrays preserved)
with open(SPLITS_PKL, "wb") as f:
    pickle.dump(splits, f)

# Save JSON (convert arrays to lists)
json_ready = [
    {
        "fold": item["fold"],
        "train_idx": item["train_idx"].tolist(),
        "test_idx": item["test_idx"].tolist(),
    }
    for item in splits
]
with open(SPLITS_JSON, "w", encoding="utf-8") as f:
    json.dump(json_ready, f, indent=2)

# Print fold sizes and checksum-like summaries
for item in splits:
    fold = item["fold"]
    test_idx = item["test_idx"]
    print(f"Fold {fold}: test size={len(test_idx)} | first 10 indices={test_idx[:10].tolist()}")

print("Saved splits to:", SPLITS_PKL, "and", SPLITS_JSON)

Fold 1: test size=56 | first 10 indices=[2, 6, 9, 11, 12, 15, 16, 18, 19, 22]
Fold 2: test size=56 | first 10 indices=[0, 3, 4, 5, 10, 23, 25, 27, 28, 32]
Fold 3: test size=55 | first 10 indices=[1, 7, 8, 13, 14, 17, 20, 21, 34, 37]
Saved splits to: splits/kfold3_seed42_indices.pkl and splits/kfold3_seed42_indices.json


# 4. Create & save true folds

In [6]:
true_fold_paths = []
for item in splits:
    fold = item["fold"]
    test_idx = item["test_idx"]
    true_fold = train_HR[test_idx]
    out_path = SPLITS_DIR / f"true_fold_{fold}.npy"
    np.save(out_path, true_fold)
    true_fold_paths.append(out_path)
    print(f"Saved true fold {fold} with shape {true_fold.shape} to {out_path}")

Saved true fold 1 with shape (56, 35778) to splits/true_fold_1.npy
Saved true fold 2 with shape (56, 35778) to splits/true_fold_2.npy
Saved true fold 3 with shape (55, 35778) to splits/true_fold_3.npy


# 5. Load predictions & match-check

In [ ]:
def load_lr_vectors(lr_path):
    if not lr_path.exists():
        raise FileNotFoundError(f"LR file not found: {lr_path}")
    df = pd.read_csv(lr_path)
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])
    return df.to_numpy()

def build_model_from_checkpoint(checkpoint_path):
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=MODEL_DEVICE)
    if isinstance(checkpoint, dict):
        state_dict = checkpoint.get("state_dict", checkpoint.get("model_state_dict", checkpoint))
        cfg = checkpoint.get("config", {})
    else:
        state_dict = checkpoint
        cfg = {}
    hidden_dims = cfg.get("hidden_dims", MODEL_HIDDEN_DIMS)
    edge_dim = cfg.get("edge_dim", MODEL_EDGE_DIM)
    dropout = cfg.get("dropout", MODEL_DROPOUT)
    model = BrainGCN_SR(hidden_dims=hidden_dims, edge_dim=edge_dim, dropout=dropout)
    model.load_state_dict(state_dict)
    model.to(MODEL_DEVICE)
    model.eval()
    return model

def predict_fold(model, lr_matrices, test_idx):
    preds = []
    with torch.no_grad():
        for i in test_idx:
            adj_lr = torch.FloatTensor(lr_matrices[i]).to(MODEL_DEVICE)
            pred_vec = model(adj_lr).clamp(min=0.0)
            preds.append(pred_vec.cpu().numpy())
    return np.array(preds)

lr_vectors = load_lr_vectors(LR_PATH)
lr_matrices = np.array([MatrixVectorizer.anti_vectorize(lr_vectors[i], 160)
                        for i in range(lr_vectors.shape[0])])

model = build_model_from_checkpoint(CHECKPOINT_PATH)

pred_folds = []
for item in splits:
    fold = item["fold"]
    test_idx = item["test_idx"]
    pred = predict_fold(model, lr_matrices, test_idx)
    pred_folds.append(pred)
    print(f"Predicted fold {fold} with shape {pred.shape}")

for item, pred in zip(splits, pred_folds):
    fold = item["fold"]
    test_idx = item["test_idx"]
    if pred.shape[0] != len(test_idx):
        raise ValueError(
            f"Fold {fold} mismatch: predictions have {pred.shape[0]} samples, "
            f"but test_idx has {len(test_idx)}. "
            "Predictions were likely generated with different folds."
        )

print("All prediction folds match the saved split indices.")

Loaded outputs/predictions_fold_1.npy with shape (2003568, 2)
Loaded outputs/predictions_fold_2.npy with shape (2003568, 2)
Loaded outputs/predictions_fold_3.npy with shape (1967790, 2)


ValueError: Fold 1 mismatch: predictions have 2003568 samples, but test_idx has 56. Predictions were likely generated with different folds.

# 6. Anti-vectorization utilities (+ quick test)

In [ ]:
def anti_vectorize_upper_triangle(vec, n=268):
    """Reconstruct a symmetric matrix from upper-triangular (i<j) column-wise vector."""
    mat = np.zeros((n, n), dtype=float)
    idx = 0
    for j in range(1, n):
        for i in range(0, j):
            mat[i, j] = vec[idx]
            mat[j, i] = vec[idx]
            idx += 1
    return mat

def vectorize_upper_triangle(mat):
    """Vectorize upper-triangular (i<j) entries column-wise."""
    n = mat.shape[0]
    vec = []
    for j in range(1, n):
        for i in range(0, j):
            vec.append(mat[i, j])
    return np.array(vec)

# Quick correctness test
rng = np.random.default_rng(0)
test_vec = rng.normal(size=(N_ROI * (N_ROI - 1)) // 2)
test_mat = anti_vectorize_upper_triangle(test_vec, n=N_ROI)
test_vec_roundtrip = vectorize_upper_triangle(test_mat)
print("Anti-vectorize/Vectorize test passed:", np.allclose(test_vec, test_vec_roundtrip))

# 7. Metric implementations

In [ ]:
def preprocess_vectors(x):
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    x = np.clip(x, 0.0, None)
    if CLIP_TO_UNIT_INTERVAL:
        x = np.clip(x, 0.0, 1.0)
    return x

def safe_prob_distribution(vec):
    v = np.clip(vec, 0.0, None)
    total = v.sum()
    if total <= 0.0:
        return np.full_like(v, 1.0 / len(v))
    return v / total

def jsd_distance(x, y):
    px = safe_prob_distribution(x + EPS)
    py = safe_prob_distribution(y + EPS)
    return jensenshannon(px, py)

def build_weighted_graph(adj):
    # Use only non-zero edges; store distance as inverse weight for shortest paths
    G = nx.from_numpy_array(adj, edge_attr="weight")
    for u, v, data in G.edges(data=True):
        w = data.get("weight", 0.0)
        data["distance"] = 1.0 / (w + EPS)
    return G

def safe_pagerank(G):
    try:
        pr = nx.pagerank(G, weight="weight")
        return np.array(list(pr.values()))
    except Exception:
        return np.zeros(G.number_of_nodes())

def safe_eigenvector_centrality(G):
    try:
        ev = nx.eigenvector_centrality(G, weight="weight", max_iter=1000)
        return np.array(list(ev.values()))
    except Exception:
        return np.zeros(G.number_of_nodes())

def safe_betweenness_centrality(G):
    try:
        bc = nx.betweenness_centrality(G, weight="distance")
        return np.array(list(bc.values()))
    except Exception:
        return np.zeros(G.number_of_nodes())

def safe_avg_clustering(G):
    try:
        return nx.average_clustering(G, weight="weight")
    except Exception:
        return 0.0

def global_efficiency_weighted(G):
    nodes = list(G.nodes())
    n = len(nodes)
    if n < 2:
        return 0.0

    total = 0.0
    count = 0
    for i in range(n):
        source = nodes[i]
        lengths = nx.single_source_dijkstra_path_length(G, source, weight="distance")
        for j in range(i + 1, n):
            target = nodes[j]
            if target in lengths:
                dist = lengths[target]
                if dist > 0:
                    total += 1.0 / dist
                    count += 1
    if count == 0:
        return 0.0
    return (2.0 * total) / (n * (n - 1))

def compute_graph_metrics_per_sample(pred_vec, true_vec):
    pred_mat = anti_vectorize_upper_triangle(pred_vec, n=N_ROI)
    true_mat = anti_vectorize_upper_triangle(true_vec, n=N_ROI)

    pred_G = build_weighted_graph(pred_mat)
    true_G = build_weighted_graph(true_mat)

    pred_pr = safe_pagerank(pred_G)
    true_pr = safe_pagerank(true_G)

    pred_ev = safe_eigenvector_centrality(pred_G)
    true_ev = safe_eigenvector_centrality(true_G)

    pred_bc = safe_betweenness_centrality(pred_G)
    true_bc = safe_betweenness_centrality(true_G)

    pred_clust = safe_avg_clustering(pred_G)
    true_clust = safe_avg_clustering(true_G)

    pred_eff = global_efficiency_weighted(pred_G)
    true_eff = global_efficiency_weighted(true_G)

    return {
        "mae_pagerank": mean_absolute_error(pred_pr, true_pr),
        "mae_eigenvector": mean_absolute_error(pred_ev, true_ev),
        "mae_betweenness": mean_absolute_error(pred_bc, true_bc),
        "abs_diff_clustering": abs(pred_clust - true_clust),
        "abs_diff_efficiency": abs(pred_eff - true_eff),
    }

# 8. Fold evaluation loop

In [ ]:
metrics_per_fold = []

for item, pred in zip(splits, pred_folds):
    fold = item["fold"]
    test_idx = item["test_idx"]
    true_fold = train_HR[test_idx]

    # Preprocess
    pred_proc = preprocess_vectors(pred)
    true_proc = preprocess_vectors(true_fold)

    # Flatten for vector-level metrics
    pred_flat = pred_proc.reshape(-1)
    true_flat = true_proc.reshape(-1)

    mae = mean_absolute_error(pred_flat, true_flat)
    pcc = pearsonr(pred_flat, true_flat)[0]
    jsd = jsd_distance(pred_flat, true_flat)

    # Graph metrics per sample
    graph_metrics_list = []
    for i in range(pred_proc.shape[0]):
        graph_metrics_list.append(
            compute_graph_metrics_per_sample(pred_proc[i], true_proc[i])
        )

    graph_df = pd.DataFrame(graph_metrics_list)
    graph_means = graph_df.mean()

    fold_result = {
        "fold": fold,
        "mae": mae,
        "pcc": pcc,
        "jsd": jsd,
        "mae_pagerank": graph_means["mae_pagerank"],
        "mae_eigenvector": graph_means["mae_eigenvector"],
        "mae_betweenness": graph_means["mae_betweenness"],
        "abs_diff_clustering": graph_means["abs_diff_clustering"],
        "abs_diff_efficiency": graph_means["abs_diff_efficiency"],
    }
    metrics_per_fold.append(fold_result)
    print(f"Fold {fold} done.")

metrics_df = pd.DataFrame(metrics_per_fold)
metrics_df

# 9. Aggregation + saving tables

In [ ]:
per_fold_path = OUTPUTS_DIR / "cv_metrics_per_fold.csv"
metrics_df.to_csv(per_fold_path, index=False)

metric_cols = [
    "mae",
    "pcc",
    "jsd",
    "mae_pagerank",
    "mae_eigenvector",
    "mae_betweenness",
    "abs_diff_clustering",
    "abs_diff_efficiency",
]

summary_mean = metrics_df[metric_cols].mean()
summary_std = metrics_df[metric_cols].std(ddof=1)

summary_df = pd.DataFrame({
    "metric": metric_cols,
    "mean": summary_mean.values,
    "std": summary_std.values,
})

summary_path = OUTPUTS_DIR / "cv_metrics_summary_mean_std.csv"
summary_df.to_csv(summary_path, index=False)

print("Saved per-fold metrics to:", per_fold_path)
print("Saved summary metrics to:", summary_path)
summary_df

# 10. Plotting + saving figures

In [ ]:
def plot_bar_with_error(summary_df, out_png, out_pdf, title=None, use_log=False):
    metrics = summary_df["metric"].tolist()
    means = summary_df["mean"].values
    stds = summary_df["std"].values

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(metrics, means, yerr=stds, capsize=4)
    ax.set_ylabel("Metric value")
    ax.set_xlabel("Metric")
    if title:
        ax.set_title(title)
    if use_log:
        ax.set_yscale("log")
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    fig.savefig(out_png, dpi=300)
    fig.savefig(out_pdf)
    plt.close(fig)

main_png = OUTPUTS_DIR / "cv_metrics_barplot.png"
main_pdf = OUTPUTS_DIR / "cv_metrics_barplot.pdf"
plot_bar_with_error(
    summary_df,
    main_png,
    main_pdf,
    title="3-Fold CV Metrics (Mean ± Std)",
    use_log=USE_LOG_SCALE
)
print("Saved main bar plots to:", main_png, "and", main_pdf)

if MAKE_SPLIT_PLOTS:
    # Example split: correlation-like vs. error-like metrics
    corr_metrics = ["pcc"]
    error_metrics = [m for m in summary_df["metric"].tolist() if m not in corr_metrics]

    corr_df = summary_df[summary_df["metric"].isin(corr_metrics)]
    err_df = summary_df[summary_df["metric"].isin(error_metrics)]

    plot_bar_with_error(
        corr_df,
        OUTPUTS_DIR / "cv_metrics_barplot_corr.png",
        OUTPUTS_DIR / "cv_metrics_barplot_corr.pdf",
        title="Correlation Metrics (Mean ± Std)",
        use_log=False
    )
    plot_bar_with_error(
        err_df,
        OUTPUTS_DIR / "cv_metrics_barplot_errors.png",
        OUTPUTS_DIR / "cv_metrics_barplot_errors.pdf",
        title="Error Metrics (Mean ± Std)",
        use_log=False
    )
    print("Saved split-scale plots to outputs/.")

# Evaluation

This notebook evaluates predicted HR connectivity matrices against ground-truth HR matrices,
using 3-fold cross-validation and graph-based metrics.

## Input

In [ ]:
fold_paths = [
    {"pred": "path/to/fold1_pred.npy", "true": "path/to/fold1_true.npy"},
    {"pred": "path/to/fold2_pred.npy", "true": "path/to/fold2_true.npy"},
    {"pred": "path/to/fold3_pred.npy", "true": "path/to/fold3_true.npy"},
]

## Imports

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from scipy.spatial.distance import jensenshannon
from scipy.stats import pearsonr
from sklearn.model_selection import KFold

## Utility functions

In [ ]:
def vector_to_adj(vec, n_nodes=268):
    """Reconstruct a symmetric adjacency matrix (n_nodes x n_nodes) from the upper triangle vector."""
    adj = np.zeros((n_nodes, n_nodes), dtype=float)
    tri_idx = np.triu_indices(n_nodes, k=1)
    adj[tri_idx] = vec
    adj = adj + adj.T
    return adj


def safe_pcc(x, y):
    """Pearson correlation with safe fallback for constant vectors."""
    if np.allclose(x, x[0]) or np.allclose(y, y[0]):
        return 0.0
    return float(pearsonr(x, y)[0])


def js_distance(p, q, eps=1e-12):
    """Jensen-Shannon distance for non-negative vectors."""
    p = np.clip(p, 0, None).astype(float)
    q = np.clip(q, 0, None).astype(float)
    p = p + eps
    q = q + eps
    p = p / np.sum(p)
    q = q / np.sum(q)
    return float(jensenshannon(p, q))


def adj_to_graph(adj, threshold=0.0):
    """Create a weighted undirected graph from an adjacency matrix."""
    adj = adj.copy()
    adj[adj < threshold] = 0.0
    return nx.from_numpy_array(adj)


def centrality_mae(cent_pred, cent_true):
    """MAE for centrality dictionaries returned by NetworkX."""
    keys = cent_true.keys()
    pred_vals = np.array([cent_pred[k] for k in keys], dtype=float)
    true_vals = np.array([cent_true[k] for k in keys], dtype=float)
    return float(np.mean(np.abs(pred_vals - true_vals)))

## Metric computation

In [ ]:
def compute_metrics_for_pair(vec_pred, vec_true, n_nodes=268, threshold=0.0):
    """Compute all requested metrics for one predicted/true pair."""
    # Vector-level metrics
    mae = float(np.mean(np.abs(vec_pred - vec_true)))
    pcc = safe_pcc(vec_pred, vec_true)
    jsd = js_distance(vec_pred, vec_true)

    # Graph-level metrics
    adj_pred = vector_to_adj(vec_pred, n_nodes=n_nodes)
    adj_true = vector_to_adj(vec_true, n_nodes=n_nodes)

    g_pred = adj_to_graph(adj_pred, threshold=threshold)
    g_true = adj_to_graph(adj_true, threshold=threshold)

    # Centralities
    pr_pred = nx.pagerank(g_pred, weight='weight')
    pr_true = nx.pagerank(g_true, weight='weight')
    ev_pred = nx.eigenvector_centrality(g_pred, weight='weight', max_iter=1000)
    ev_true = nx.eigenvector_centrality(g_true, weight='weight', max_iter=1000)
    bc_pred = nx.betweenness_centrality(g_pred, weight='weight', normalized=True)
    bc_true = nx.betweenness_centrality(g_true, weight='weight', normalized=True)

    pr_mae = centrality_mae(pr_pred, pr_true)
    ev_mae = centrality_mae(ev_pred, ev_true)
    bc_mae = centrality_mae(bc_pred, bc_true)

    # Clustering coefficient and global efficiency
    clust_pred = nx.average_clustering(g_pred, weight='weight')
    clust_true = nx.average_clustering(g_true, weight='weight')
    clust_diff = float(np.abs(clust_pred - clust_true))

    eff_pred = nx.global_efficiency(g_pred)
    eff_true = nx.global_efficiency(g_true)
    eff_diff = float(np.abs(eff_pred - eff_true))

    return {
        'MAE': mae,
        'PCC': pcc,
        'JSD': jsd,
        'PageRank_MAE': pr_mae,
        'Eigenvector_MAE': ev_mae,
        'Betweenness_MAE': bc_mae,
        'Clustering_Coeff_Diff': clust_diff,
        'Global_Efficiency_Diff': eff_diff
    }

## Cross-validation evaluation

In [ ]:
# Ground-truth splits for 3-fold CV
# Load your training HR data (shape: [167, 35778])
# Example: train_HR = np.load('train_HR.npy')
train_HR = None

if train_HR is None:
    raise ValueError("Please load train_HR before running this cell.")

kf = KFold(n_splits=3, shuffle=True, random_state=42)
for fold_idx, (_, test_idx) in enumerate(kf.split(train_HR), start=1):
    true_fold = train_HR[test_idx]
    np.save(f"true_fold_{fold_idx}.npy", true_fold)
    print(f"Fold {fold_idx} | true_fold shape: {true_fold.shape}")
    print(f"Indices: {test_idx}")


In [ ]:
predicted_HR = None
ground_truth_HR = None

# Quick shape validation helper
def validate_inputs(predicted, ground_truth, n_nodes=268):
    if predicted is None or ground_truth is None:
        raise ValueError('Please load predicted_HR and ground_truth_HR before running.')
    if predicted.shape != ground_truth.shape:
        raise ValueError('Predicted and ground-truth arrays must have the same shape.')
    if predicted.ndim != 2:
        raise ValueError('Expected 2D arrays of shape (N, 35778).')
    if predicted.shape[1] != (n_nodes * (n_nodes - 1)) // 2:
        raise ValueError('Second dimension does not match upper triangle size.')


def evaluate_cross_validation(predicted, ground_truth, n_splits=3, random_state=42):
    validate_inputs(predicted, ground_truth)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    fold_metrics = []
    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(predicted), start=1):
        # Evaluate on the held-out fold
        metrics_list = []
        for i in test_idx:
            m = compute_metrics_for_pair(predicted[i], ground_truth[i])
            metrics_list.append(m)
        fold_df = pd.DataFrame(metrics_list)
        fold_summary = fold_df.mean().to_dict()
        fold_summary['fold'] = fold_idx
        fold_metrics.append(fold_summary)
        print(f'Fold {fold_idx} done: {len(test_idx)} samples')

    fold_metrics_df = pd.DataFrame(fold_metrics).set_index('fold')
    return fold_metrics_df

## Visualization (bar plots)

In [ ]:
def plot_cv_bars(fold_metrics_df, title='3-Fold CV Metrics'):
    metrics = fold_metrics_df.columns.tolist()
    means = fold_metrics_df.mean(axis=0)
    stds = fold_metrics_df.std(axis=0)

    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(metrics))
    ax.bar(x, means, yerr=stds, capsize=4, color='#4C78A8', alpha=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels(metrics, rotation=30, ha='right')
    ax.set_ylabel('Metric Value')
    ax.set_title(title)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

In [ ]:
def evaluate_from_fold_paths(fold_paths):
    fold_metrics = []
    for fold_idx, paths in enumerate(fold_paths, start=1):
        pred = np.load(paths["pred"])
        true = np.load(paths["true"])
        validate_inputs(pred, true)
        metrics_list = [compute_metrics_for_pair(pred[i], true[i]) for i in range(pred.shape[0])]
        fold_df = pd.DataFrame(metrics_list)
        fold_summary = fold_df.mean().to_dict()
        fold_summary["fold"] = fold_idx
        fold_metrics.append(fold_summary)
        print(f"Fold {fold_idx} done: {pred.shape[0]} samples")

    return pd.DataFrame(fold_metrics).set_index("fold")

fold_metrics_df = evaluate_from_fold_paths(fold_paths)
plot_cv_bars(fold_metrics_df)